# 04d – Haladó CNN-ek (ResNet-50 & EfficientNet-B3)

**Szerző:** Magda Ferenc (U5O0BB)  
**Projekt:** Gitár-akkord felismerő szoftver gépi látással  
**Notebook célja:** Nagyobb kapacitású CNN architektúrák fine-tuningja.

**Modellek:**
1. ResNet-50 (~25M param, klasszikus, jól dokumentált)
2. EfficientNet-B3 (~12M param, jobb acc/param arány mint B0)

---

## Notebook sorozat

| Notebook | Modell | Státusz |
|----------|--------|---------|
| `04a` | Hagyományos ML | kész |
| `04b` | MobileNetV3, ShuffleNetV2 | kész |
| `04c` | EfficientNet-B0 | kész |
| **`04d`** | **ResNet-50, EfficientNet-B3** | **← jelen** |
| `04e` | ViT-B/16 (opcionális) | tervezett |

## Tartalomjegyzék
1. Konfiguráció  
2. DataLoader  
3. Training utilities  
4. ResNet-50  
5. EfficientNet-B3  
6. Összehasonlítás az összes modellel  
7. Összefoglaló


## 1. Konfiguráció

In [ ]:
import warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report)

warnings.filterwarnings("ignore")

NOTEBOOK_DIR   = Path.cwd()
PROJECT_ROOT   = NOTEBOOK_DIR.parent
DATA_ROOT      = PROJECT_ROOT / "data"
MANIFEST_PATH  = DATA_ROOT / "split_manifest.csv"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR     = PROJECT_ROOT / "output" / "04d_advanced_cnn"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_WORKERS = 0
RANDOM_SEED = 42
LR_A, EPOCHS_A, PATIENCE_A = 1e-3, 20, 8
LR_B_HEAD, LR_B_BB         = 1e-4, 1e-5
EPOCHS_B, PATIENCE_B        = 30, 10
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
print(f"Eszkoz: {DEVICE}")


## 2. DataLoader

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
CLASSES     = sorted(manifest["class"].unique())
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(CLASSES)
manifest["label"] = manifest["class"].map(CLASS2IDX)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df   = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df  = manifest[manifest["split"] == "test"].reset_index(drop=True)

class_counts  = train_df["class"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / (NUM_CLASSES * class_counts.get(c, 1)) for c in CLASSES],
    dtype=torch.float
).to(DEVICE)

train_tf = T.Compose([T.Resize(256), T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(0.5), T.ColorJitter(0.3, 0.3, 0.2, 0.1),
    T.RandomRotation(15), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
valtest_tf = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class GuitarChordDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df, self.transform = df.reset_index(drop=True), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(row["label"])

train_loader = DataLoader(GuitarChordDataset(train_df, train_tf),
    BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(GuitarChordDataset(val_df, valtest_tf),
    BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(GuitarChordDataset(test_df, valtest_tf),
    BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")


## 3. Training utilities

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, ckpt_path=None):
        self.patience, self.min_delta = patience, min_delta
        self.ckpt_path = ckpt_path
        self.best_loss, self.counter, self.best_state = float("inf"), 0, None
    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if self.ckpt_path: torch.save(self.best_state, self.ckpt_path)
        else: self.counter += 1
        return self.counter >= self.patience
    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f"  Legjobb visszatoltve (val_loss={self.best_loss:.4f})")

def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = phase == "train"
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs); loss = criterion(logits, labels)
            if is_train: optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()*len(labels)
            correct    += (logits.argmax(1)==labels).sum().item()
            total      += len(labels)
    return total_loss/total, correct/total

def two_phase_train(model, label, freeze_fn, unfreeze_fn):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    freeze_fn(model)
    optA = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_A, weight_decay=1e-4)
    schA = CosineAnnealingLR(optA, T_max=EPOCHS_A, eta_min=1e-5)
    esA  = EarlyStopping(PATIENCE_A, ckpt_path=CHECKPOINT_DIR/f"best_{label}_A.pth")
    t0 = time.time()
    print(f"[{label}] Fazis A")
    for ep in range(1, EPOCHS_A+1):
        tl, ta = run_epoch(model, train_loader, criterion, optA, "train")
        vl, va = run_epoch(model, val_loader,   criterion, None, "val")
        schA.step()
        print(f"  Ep{ep:>2}  tl={tl:.4f}  ta={ta:.3f}  vl={vl:.4f}  va={va:.3f} ({time.time()-t0:.0f}s)")
        if esA.step(vl, model): print("  Early stop."); break
    esA.restore_best(model)

    unfreeze_fn(model)
    param_groups = []
    if hasattr(model, "layer4"): # ResNet
        param_groups = [
            {"params": list(model.layer3.parameters())+list(model.layer4.parameters()), "lr": LR_B_BB},
            {"params": list(model.fc.parameters()), "lr": LR_B_HEAD},
        ]
    elif hasattr(model, "features"): # EfficientNet
        param_groups = [
            {"params": model.features.parameters(), "lr": LR_B_BB},
            {"params": model.classifier.parameters(), "lr": LR_B_HEAD},
        ]
    else:
        param_groups = [{"params": filter(lambda p: p.requires_grad, model.parameters()), "lr": LR_B_HEAD}]

    optB = AdamW(param_groups, weight_decay=1e-4)
    schB = CosineAnnealingLR(optB, T_max=EPOCHS_B, eta_min=1e-6)
    esB  = EarlyStopping(PATIENCE_B, ckpt_path=CHECKPOINT_DIR/f"best_{label}_B.pth")
    print(f"[{label}] Fazis B")
    for ep in range(1, EPOCHS_B+1):
        tl, ta = run_epoch(model, train_loader, criterion, optB, "train")
        vl, va = run_epoch(model, val_loader,   criterion, None, "val")
        schB.step()
        print(f"  Ep{ep:>2}  tl={tl:.4f}  ta={ta:.3f}  vl={vl:.4f}  va={va:.3f} ({time.time()-t0:.0f}s)")
        if esB.step(vl, model): print("  Early stop."); break
    esB.restore_best(model)

def evaluate(model, label):
    model.eval(); preds, labs = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().tolist())
            labs.extend(labels.tolist())
    acc = accuracy_score(labs, preds)
    f1  = f1_score(labs, preds, average="macro")
    print(f"[{label}] Test acc={acc:.4f}  Macro F1={f1:.4f}")
    print(classification_report(labs, preds, target_names=CLASSES, digits=3))
    return acc, f1, preds, labs

ALL_RESULTS_D = []
print("Utils D betoltve.")


## 4. ResNet-50

~25M paraméter, skip-connection architektúra, kiforrott baseline CNN. Layer3 + Layer4 felolvad Fázis B-ben.

In [ ]:
# ResNet-50
def freeze_resnet(model):
    for p in model.parameters(): p.requires_grad = False
    for p in model.fc.parameters(): p.requires_grad = True

def unfreeze_resnet(model):
    for p in model.layer3.parameters(): p.requires_grad = True
    for p in model.layer4.parameters(): p.requires_grad = True
    for p in model.fc.parameters():     p.requires_grad = True

resnet50 = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
resnet50.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(resnet50.fc.in_features, NUM_CLASSES)
)
resnet50 = resnet50.to(DEVICE)

total = sum(p.numel() for p in resnet50.parameters())
print(f"ResNet-50  osszes param: {total:,}")

two_phase_train(resnet50, "ResNet50", freeze_resnet, unfreeze_resnet)
acc_r50, f1_r50, preds_r50, labs_r50 = evaluate(resnet50, "ResNet50")
ALL_RESULTS_D.append({"Model": "ResNet-50", "Test Acc": acc_r50, "Macro F1": f1_r50})
torch.save(resnet50.state_dict(), CHECKPOINT_DIR / "resnet50.pth")


## 5. EfficientNet-B3

~12M paraméter, 3.6× hatékonyabb mint B0. Compound scaling – szélesség, mélység, felbontás egyszerre skálázva. Utolsó 4 blokk felolvad Fázis B-ben.

In [ ]:
# EfficientNet-B3
def freeze_eff(model):
    for p in model.features.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True

def unfreeze_eff(model):
    blocks = list(model.features.children())
    for blk in blocks[-4:]:
        for p in blk.parameters(): p.requires_grad = True

effb3 = tv_models.efficientnet_b3(weights=tv_models.EfficientNet_B3_Weights.DEFAULT)
in_features = effb3.classifier[1].in_features
effb3.classifier = nn.Sequential(
    nn.Dropout(p=0.35, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)
effb3 = effb3.to(DEVICE)

total = sum(p.numel() for p in effb3.parameters())
print(f"EfficientNet-B3  osszes param: {total:,}")

two_phase_train(effb3, "EffB3", freeze_eff, unfreeze_eff)
acc_b3, f1_b3, preds_b3, labs_b3 = evaluate(effb3, "EffB3")
ALL_RESULTS_D.append({"Model": "EfficientNet-B3", "Test Acc": acc_b3, "Macro F1": f1_b3})
torch.save(effb3.state_dict(), CHECKPOINT_DIR / "efficientnet_b3.pth")


## 6. Teljes összehasonlítás (04a–04d)

In [ ]:
# Összes notebook eredményének összefüggésben való megjelenítése
# (ha a CSV-k elérhetők a korábbi notebookokból)

results_d_df = pd.DataFrame(ALL_RESULTS_D).sort_values("Test Acc", ascending=False)
print("=== 04d Leaderboard ===")
print(results_d_df.to_string(index=False))
results_d_df.to_csv(OUTPUT_DIR / "advanced_cnn_results.csv", index=False)

# Összegyűjti az összes notebook eredményét, ha elérhetők
all_csvs = {
    "04a Baseline ML":  PROJECT_ROOT / "output" / "04a_baseline_ml" / "baseline_results.csv",
    "04b Mobile CNN":   PROJECT_ROOT / "output" / "04b_mobile_cnn"  / "mobile_cnn_results.csv",
    "04c EfficientB0":  None,  # notebook-ból kell átmásolni manuálisan
    "04d Advanced CNN": OUTPUT_DIR / "advanced_cnn_results.csv",
}

combined_rows = []
for label, csv_path in all_csvs.items():
    if csv_path and csv_path.exists():
        df = pd.read_csv(csv_path)[["Model", "Test Acc"]]
        df["Notebook"] = label
        combined_rows.append(df)

if combined_rows:
    combined_df = pd.concat(combined_rows).sort_values("Test Acc", ascending=True)
    fig, ax = plt.subplots(figsize=(12, max(5, len(combined_df)*0.45)))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(combined_df)))
    bars = ax.barh(combined_df["Model"], combined_df["Test Acc"], color=colors)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Test Accuracy")
    ax.set_title("Osszes modell – Test Accuracy osszehasonlitas")
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, combined_df["Test Acc"]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "full_model_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()
    combined_df.sort_values("Test Acc", ascending=False).to_csv(
        OUTPUT_DIR / "full_model_comparison.csv", index=False)
    print("Teljes összehasonlítás elmentve.")
else:
    print("Más notebookok CSV-jei még nem elérhetők. Futtasd 04a és 04b notebookokat előbb.")


## 7. Összefoglaló és következő lépések

### Mit értünk el?
- ResNet-50 és EfficientNet-B3 fine-tuning kétfázisú stratégiával
- Teljes modell-összehasonlítás (04a–04d) vizualizálva és CSV-ben rögzítve
- Checkpointok mentve: `checkpoints/resnet50.pth`, `efficientnet_b3.pth`

### Várható sorrend (teljes sorozat)
| Rang | Modell | Várható Test Acc |
|------|--------|-----------------|
| 1 | EfficientNet-B3 | ~90–94% |
| 2 | ResNet-50 | ~88–92% |
| 3 | EfficientNet-B0 | ~85–92% |
| 4 | MobileNetV3-Large | ~80–88% |
| 5 | SVM + CNN features | ~75–82% |
| 6 | MobileNetV3-Small | ~78–85% |
| 7 | ShuffleNetV2 | ~75–83% |
| 8–10 | HOG-alapú ML modellek | ~60–78% |

> A kis adathalmaz (207 train kép) miatt a rangsor varianciája nagy.
> Valós eredmények eltérhetnek a fenti becsléstől.

### Következő lépések – `04e_vit.ipynb` (opcionális)

Vision Transformer (ViT-B/16) fine-tuning. Csak akkor ajánlott, ha:
- A GPU legalább 6 GB VRAM-mal rendelkezik
- A legjobb CNN eredmény < 90%

### Vagy: `05_model_selection.ipynb`
Ensemble + végső modell kiválasztás az összes notebook eredménye alapján.
